# LinQ Safety: падения, новый временной RGB-классификатор

**Обучать одновременно с PPE**, на CPU — не занимать два T4 основного задания.
Новая задача от ImageNet ResNet18 + GRU, без прежних task checkpoints.
2 083 исходных RGB-видео: 2 013 уникальных Figshare и 70 прежних URFD.
Figshare разделён по участникам (20 train / 4 val / 5 test); URFD сохраняет прежний
video split, независимость людей в URFD не доказана. Начало падения размечено лишь
в URFD: Figshare даёт метрики **классификации клипов**, не event F1 и не задержку.

До 30 эпох temporal head, затем до 4 эпох / 30 минут дообучения layer4.
Ограничение CPU-задачи 100 минут, общий предел ночи **11:15 МСК 24 октября**.
Checkpoints каждой эпохи, optimizer, RNG, предыдущая копия; приватный внешний backup.
Frame encoder экспортируется отдельно для LinQ ResNet18; GRU выполняется на CPU.
Новые веса не устанавливаются в сайт без независимой проверки.

In [ ]:
import time,sys,subprocess
from datetime import datetime
print('LINQ_FALL_WORKER_BOOTED',time.time(),flush=True)
STOP_UTC='2026-10-24T08:15:00+00:00'
remaining=(datetime.fromisoformat(STOP_UTC).timestamp()-time.time())/60-10
assert remaining>20,'The approved overnight deadline has passed.'
CPU_BUDGET=min(100,remaining)
subprocess.run([sys.executable,'-m','pip','install','--quiet',
 'onnx==1.19.1','protobuf==5.29.5','onnxruntime==1.29.0','kagglehub'],check=True)
from pathlib import Path
import json,shutil,zipfile,hashlib
WORK=Path('/kaggle/working');OUTPUT=WORK/'fall-training-output';OUTPUT.mkdir(exist_ok=True)
SCRIPTS=WORK/'scripts';SCRIPTS.mkdir(exist_ok=True)
APP=WORK/'app';APP.mkdir(exist_ok=True);(APP/'__init__.py').write_text('')

In [ ]:
(SCRIPTS/'train_fall_temporal.py').write_text('"""Continue the evaluated RGB encoder with clip-supervised temporal fall training.\n\nAll model/threshold selection uses validation. Test is read only for final scoring.\nFigshare has action labels, not onset times: do not call its clip F1 event F1.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset,DataLoader,TensorDataset,WeightedRandomSampler\n\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT))\nfrom app.fall_temporal import TemporalFallNet\nfrom training_checkpoint import atomic_torch_save\n\n\nclass Clips(Dataset):\n    def __init__(self,root,rows,augment=False):self.root=root;self.rows=rows;self.augment=augment\n    def __len__(self):return len(self.rows)\n    def __getitem__(self,index):\n        row=self.rows[index];frames=[]\n        flip=self.augment and torch.rand(()).item()<.5\n        brightness=(.85+.3*torch.rand(()).item()) if self.augment else 1\n        # The same transform is used throughout one clip; never shuffle its frames.\n        for path in sorted((self.root/row[\'frames\']).glob(\'*.jpg\')):\n            with Image.open(path) as image:x=np.asarray(image.convert(\'RGB\'),dtype=np.float32).transpose(2,0,1)/255\n            if flip:x=x[:,:,::-1].copy()\n            x=np.clip(x*brightness,0,1)\n            frames.append((x-np.array([.485,.456,.406],np.float32)[:,None,None])/\n                          np.array([.229,.224,.225],np.float32)[:,None,None])\n        if len(frames)!=16:raise ValueError(\'Every clip must contain exactly 16 RGB frames\')\n        return torch.from_numpy(np.stack(frames)),row[\'label\']\n\n\ndef stats(scores,labels,threshold=.5):\n    predicted=np.asarray(scores)>=threshold;truth=np.asarray(labels).astype(bool)\n    tp=int((predicted&truth).sum());fp=int((predicted&~truth).sum());fn=int((~predicted&truth).sum());tn=int((~predicted&~truth).sum())\n    return {\'precision\':tp/max(tp+fp,1),\'recall\':tp/max(tp+fn,1),\'f1\':2*tp/max(2*tp+fp+fn,1),\n            \'tp\':tp,\'fp\':fp,\'fn\':fn,\'tn\':tn}\n\n\ndef train(args):\n    torch.set_num_threads(args.cpu_threads);torch.manual_seed(74291)\n    device=torch.device(args.device);output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    data=Path(args.data);manifest=data/\'fall-manifest.jsonl\';manifest_sha=hashlib.sha256(manifest.read_bytes()).hexdigest()\n    rows=[json.loads(line) for line in manifest.read_text(encoding=\'utf-8\').splitlines()]\n    parts={name:[r for r in rows if r[\'split\']==name] for name in [\'train\',\'val\',\'test\']}\n    if not all(len(parts[k])>=8 for k in parts):raise ValueError(\'Insufficient independent clips\')\n    model=TemporalFallNet()\n    if args.fresh:\n        from torchvision.models import ResNet18_Weights\n        initial_state=ResNet18_Weights.DEFAULT.get_state_dict(progress=True,check_hash=True)\n        initial_encoder=\'official ImageNet ResNet18; no previous fall checkpoint\'\n    else:\n        initial_state=torch.load(data/\'initial-fall-resnet18.pt\',map_location=\'cpu\',weights_only=True)[\'state_dict\']\n        initial_encoder=hashlib.sha256((data/\'initial-fall-resnet18.pt\').read_bytes()).hexdigest()\n    model.encoder.load_state_dict({k:v for k,v in initial_state.items() if not k.startswith(\'fc.\')},strict=True)\n    model.to(device)\n    last=output/\'fall-temporal-last.pt\';best_path=output/\'fall-temporal.pt\'\n    prior=torch.load(last,map_location=\'cpu\',weights_only=True) if args.resume and last.exists() else None\n    if prior:\n        if prior[\'data_manifest_sha256\']!=manifest_sha:raise ValueError(\'Resume corpus/split identity changed\')\n        model.load_state_dict(prior[\'state_dict\']);torch.set_rng_state(prior[\'torch_rng\'])\n    start=time.monotonic();deadline=start+args.max_minutes*60\n    best=float(prior.get(\'best_validation_f1\',-1)) if prior else -1\n    loaders={part:DataLoader(Clips(data,part_rows),batch_size=args.batch,shuffle=False,num_workers=args.workers)\n             for part,part_rows in parts.items() if part!=\'test\'}\n\n    def evaluate(loader,features=False):\n        model.eval();scores=[];labels=[]\n        with torch.inference_mode():\n            for x,y in loader:\n                logits=model.classify_features(x.to(device)) if features else model(x.to(device))\n                scores.extend(logits.softmax(1)[:,1].cpu().tolist());labels.extend(y.tolist())\n        return np.asarray(scores),np.asarray(labels)\n\n    def checkpoint(stage,epoch,optimizer,partial=False):\n        state={\'architecture\':\'resnet18-gru-v1\',\'state_dict\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n            \'stage\':stage,\'epoch\':epoch,\'partial_epoch\':partial,\'best_validation_f1\':best,\n            \'data_manifest_sha256\':manifest_sha,\'initial_encoder\':initial_encoder,\n            \'torch_rng\':torch.get_rng_state(),\'classes\':[\'no_fall\',\'fall\'],\'frames\':16,\'window_seconds\':2.0,\n            \'threshold\':.5,\'evaluation_unit\':\'action clip, not incident onset\'}\n        atomic_torch_save(state,last,previous=True)\n        print(\'FALL_CHECKPOINT_SAVED\',stage,\'epoch\',epoch,\'bytes\',last.stat().st_size,flush=True)\n        return state\n\n    # Freeze and cache RGB features: the temporal stage is cheap even on CPU.\n    feature_loaders={};model.eval()\n    for part in [\'train\',\'val\']:\n        feature_path=output/(\'features-\'+part+\'.pt\')\n        if feature_path.exists():\n            cached=torch.load(feature_path,map_location=\'cpu\',weights_only=True)\n            if cached[\'data_manifest_sha256\']!=manifest_sha:raise ValueError(\'Feature cache identity differs\')\n            vectors,targets=cached[\'features\'],cached[\'labels\']\n        else:\n            features=[];targets=[]\n            with torch.inference_mode():\n                for batch,(x,y) in enumerate(loaders[part]):\n                    b,t,c,h,w=x.shape\n                    encoded=model.encode(x.reshape(b*t,c,h,w).to(device)).reshape(b,t,512)\n                    features.append(encoded.cpu());targets.append(y)\n                    if batch%25==0:print(\'FALL_FEATURES\',part,batch,\'/\',len(loaders[part]),flush=True)\n                    if time.monotonic()>deadline-15*60:raise TimeoutError(\'Feature extraction consumed the budget; partial caches/checkpoints retained\')\n            vectors=torch.cat(features);targets=torch.cat(targets)\n            atomic_torch_save({\'features\':vectors,\'labels\':targets,\'data_manifest_sha256\':manifest_sha},feature_path)\n        dataset=TensorDataset(vectors,targets)\n        counts=torch.bincount(targets,minlength=2).float()\n        sampler=WeightedRandomSampler((1/counts)[targets],len(targets),replacement=True) if part==\'train\' else None\n        feature_loaders[part]=DataLoader(dataset,batch_size=32,sampler=sampler)\n\n    history=[];criterion=nn.CrossEntropyLoss()\n    for stage,epochs,phase_minutes in [(\'head\',args.head_epochs,12),(\'finetune\',args.finetune_epochs,args.finetune_minutes)]:\n        if prior and prior[\'stage\']==\'finetune\' and stage==\'head\':continue\n        for name,p in model.named_parameters():p.requires_grad=not name.startswith(\'encoder.\') or (stage==\'finetune\' and name.startswith(\'encoder.layer4.\'))\n        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=3e-4 if stage==\'head\' else 1e-5,weight_decay=1e-3)\n        first=0\n        if prior and prior[\'stage\']==stage:\n            optimizer.load_state_dict(prior[\'optimizer\']);first=prior[\'epoch\'];print(\'FALL_RESTORED\',stage,first,flush=True)\n        loader=feature_loaders[\'train\'] if stage==\'head\' else DataLoader(Clips(data,parts[\'train\'],True),\n            batch_size=args.batch,shuffle=True,num_workers=args.workers)\n        phase_deadline=min(deadline-10*60,time.monotonic()+phase_minutes*60)\n        for epoch in range(first,epochs):\n            if time.monotonic()>=phase_deadline:break\n            model.train();model.encoder.eval()\n            if stage==\'finetune\':model.encoder.layer4.train()\n            loss_sum=0;partial=False\n            for batch,(x,y) in enumerate(loader):\n                optimizer.zero_grad(set_to_none=True);x,y=x.to(device),y.to(device)\n                logits=model.classify_features(x) if stage==\'head\' else model(x)\n                loss=criterion(logits,y);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5);optimizer.step()\n                loss_sum+=float(loss.detach())\n                if time.monotonic()>=phase_deadline:partial=batch+1<len(loader);break\n            scores,labels=evaluate(feature_loaders[\'val\'] if stage==\'head\' else loaders[\'val\'],features=stage==\'head\')\n            metrics=stats(scores,labels);improved=metrics[\'f1\']>best\n            if improved:best=metrics[\'f1\']\n            state=checkpoint(stage,epoch if partial else epoch+1,optimizer,partial)\n            if improved:atomic_torch_save(state,best_path)\n            row={\'stage\':stage,\'epoch\':epoch+1,\'partial_epoch\':partial,\'train_loss\':loss_sum/max(batch+1,1),\n                 \'validation_clip\':metrics,\'elapsed_seconds\':time.monotonic()-start}\n            history.append(row);(output/\'fall-training-history.json\').write_text(json.dumps(history,indent=2))\n            print(\'FALL_VALIDATION\',json.dumps(row),flush=True)\n            if partial:break\n    saved=torch.load(best_path,map_location=\'cpu\',weights_only=True);model.load_state_dict(saved[\'state_dict\']);model.to(device)\n    scores,labels=evaluate(loaders[\'val\'])\n    candidates=[(stats(scores,labels,float(t)),float(t)) for t in np.linspace(.05,.95,91)]\n    _,threshold=max(candidates,key=lambda pair:(pair[0][\'f1\'],pair[0][\'precision\'],pair[1]))\n    saved[\'threshold\']=threshold;atomic_torch_save(saved,best_path)\n    test_loader=DataLoader(Clips(data,parts[\'test\']),batch_size=args.batch,num_workers=args.workers)\n    test_scores,test_labels=evaluate(test_loader)\n    report={\'architecture\':\'resnet18-gru-v1\',\'threshold_selected_on_validation\':threshold,\n        \'validation_clip\':stats(scores,labels,threshold),\'test_clip\':stats(test_scores,test_labels,threshold),\n        \'test_by_source\':{source:stats(test_scores[[i for i,r in enumerate(parts[\'test\']) if r[\'source\']==source]],\n            test_labels[[i for i,r in enumerate(parts[\'test\']) if r[\'source\']==source]],threshold) for source in {r[\'source\'] for r in parts[\'test\']}},\n        \'data_manifest_sha256\':manifest_sha,\'split_counts\':{k:len(v) for k,v in parts.items()},\n        \'elapsed_minutes\':(time.monotonic()-start)/60,\'event_timing_metrics\':\'not measured on Figshare: onset labels unavailable\',\n        \'linq_encoder\':\'ResNet18 family; new weights require compilation and hardware test\',\n        \'temporal_head\':\'small GRU on CPU; no LinQ GRU support claimed\',\n        \'deployment\':\'candidate only; do not replace the evaluated baseline automatically\'}\n    (output/\'fall-temporal-metrics.json\').write_text(json.dumps(report,indent=2))\n    print(\'FALL_TEST\',json.dumps(report),flush=True)\n    # Split export keeps the supported ResNet encoder separate from temporal CPU inference.\n    model.eval().cpu()\n    torch.onnx.export(model.encoder,torch.zeros(1,3,224,224),str(output/\'fall-temporal-encoder.onnx\'),\n        input_names=[\'images\'],output_names=[\'features\'],opset_version=12,dynamo=False)\n    class Head(nn.Module):\n        def __init__(self,network):super().__init__();self.network=network\n        def forward(self,features):return self.network.classify_features(features)\n    torch.onnx.export(Head(model),torch.zeros(1,16,512),str(output/\'fall-temporal-head.onnx\'),\n        input_names=[\'features\'],output_names=[\'logits\'],opset_version=12,dynamo=False)\n    print(\'FALL_TRAINING_COMPLETE\',flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--device\',default=\'cpu\');p.add_argument(\'--cpu-threads\',type=int,default=4)\n    p.add_argument(\'--batch\',type=int,default=4);p.add_argument(\'--workers\',type=int,default=2)\n    p.add_argument(\'--head-epochs\',type=int,default=30);p.add_argument(\'--finetune-epochs\',type=int,default=4)\n    p.add_argument(\'--finetune-minutes\',type=float,default=25);p.add_argument(\'--max-minutes\',type=float,default=90)\n    p.add_argument(\'--resume\',action=\'store_true\');p.add_argument(\'--fresh\',action=\'store_true\');train(p.parse_args())\n',encoding='utf-8')
(SCRIPTS/'training_checkpoint.py').write_text('"""Crash-safe replacement of our own training checkpoints on one filesystem."""\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef atomic_copy(source, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with Path(source).open(\'rb\') as incoming, temporary.open(\'wb\') as outgoing:\n        shutil.copyfileobj(incoming, outgoing)\n        outgoing.flush()\n        os.fsync(outgoing.fileno())\n    os.replace(temporary, target)\n\n\ndef atomic_torch_save(state, target, previous=False):\n    import torch\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    if previous and target.exists():\n        atomic_copy(target, target.with_name(target.stem + \'-previous.pt\'))\n    os.replace(temporary, target)\n',encoding='utf-8')
(SCRIPTS/'backup_training.py').write_text('"""Optional private Kaggle Dataset backup using the official authenticated SDK.\n\nNo credentials are read or printed by this script. A lack of write authentication\nis reported honestly; local epoch checkpoints remain available independently.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport zipfile\n\nfrom training_checkpoint import atomic_copy\n\n\ndef snapshot(root, destination):\n    files=[]\n    for source in sorted(root.rglob(\'*\')):\n        if not source.is_file():continue\n        if source.suffix not in {\'.pt\',\'.json\',\'.csv\',\'.yaml\',\'.log\'}:continue\n        if source.name.endswith((\'-previous.pt\',\'resume-previous.pt\')):continue\n        target=destination/source.relative_to(root)\n        atomic_copy(source,target)\n        if target.suffix==\'.pt\':\n            try:\n                with zipfile.ZipFile(target) as archive:\n                    if archive.testzip() is not None:raise ValueError(\'Checkpoint CRC mismatch\')\n            except (zipfile.BadZipFile,ValueError,EOFError):\n                target.unlink();continue  # A non-atomic library best/last write may still be in progress.\n        files.append({\'path\':str(source.relative_to(root)),\n                      \'sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\'bytes\':target.stat().st_size})\n    (destination/\'backup-manifest.json\').write_text(json.dumps({\'created_utc\':time.time(),\'files\':files},indent=2))\n    return files\n\n\ndef upload_once(root, handle, probe=False):\n    with tempfile.TemporaryDirectory(prefix=\'linq-checkpoint-backup-\') as name:\n        destination=Path(name)\n        if probe:\n            (destination/\'backup-probe.json\').write_text(json.dumps({\'purpose\':\'private LinQ training checkpoint backup\',\'time\':time.time()}))\n        else:snapshot(root,destination)\n        # Native authentication may support downloads without write access.\n        # Do not call login() or silently request/create an API credential.\n        import kagglehub\n        kagglehub.dataset_upload(handle,str(destination),version_notes=\'LinQ checkpoint snapshot; optimizer and best weights\')\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--handle\',required=True)\n    p.add_argument(\'--once\',action=\'store_true\');p.add_argument(\'--probe\',action=\'store_true\')\n    p.add_argument(\'--interval\',type=int,default=1800)\n    args=p.parse_args();root=Path(args.root);root.mkdir(parents=True,exist_ok=True)\n    status=root/\'remote-backup-status.json\'\n    if args.once:\n        try:\n            upload_once(root,args.handle,args.probe)\n            status.write_text(json.dumps({\'verified_upload\':True,\'handle\':args.handle,\'time\':time.time(),\'probe\':args.probe}))\n            print(\'REMOTE_BACKUP_UPLOAD_CONFIRMED\',args.handle,flush=True)\n        except Exception as error:\n            status.write_text(json.dumps({\'verified_upload\':False,\'handle\':args.handle,\'time\':time.time(),\n                \'error_type\':type(error).__name__,\'warning\':\'Local files alone do not survive a destroyed VM.\'}))\n            print(\'REMOTE_BACKUP_UNAVAILABLE\',type(error).__name__,\'No credential or response body logged.\',flush=True)\n            raise SystemExit(2)\n        return\n    seen=set();last_upload=0\n    while True:\n        checkpoints={str(p.relative_to(root)) for p in root.rglob(\'*.pt\')\n                     if p.name==\'resume-last.pt\' or p.name.endswith((\'classifier-last.pt\',\'temporal-last.pt\'))}\n        if not checkpoints or (not(checkpoints-seen) and time.monotonic()-last_upload<args.interval):\n            time.sleep(20);continue\n        command=[sys.executable,__file__,\'--root\',str(root),\'--handle\',args.handle,\'--once\']\n        try:\n            result=subprocess.run(command,timeout=300,check=False)\n            if result.returncode==0:seen=checkpoints;last_upload=time.monotonic()\n        except subprocess.TimeoutExpired:print(\'REMOTE_BACKUP_TIMEOUT; training continues\',flush=True)\n        time.sleep(20 if last_upload else 60)\n\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
(APP/'fall_temporal.py').write_text('"""RGB temporal model. ResNet18 frame encoder + a small CPU-capable GRU head.\n\nInputs are RGB frames in chronological order over a two-second window.\nClip classification is not itself validated incident onset localization.\n"""\nfrom collections import deque\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import resnet18\n\n\nclass TemporalFallNet(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.encoder=resnet18(weights=None);self.encoder.fc=nn.Identity()\n        self.temporal=nn.GRU(1024,128,batch_first=True)\n        self.classifier=nn.Sequential(nn.LayerNorm(256),nn.Linear(256,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,2))\n\n    def encode(self,frames):return self.encoder(frames)\n\n    def classify_features(self,features):\n        features=F.layer_norm(features,(512,))\n        delta=torch.cat([torch.zeros_like(features[:,:1]),features[:,1:]-features[:,:-1]],dim=1)\n        output,_=self.temporal(torch.cat([features,delta],dim=-1))\n        return self.classifier(torch.cat([output[:,-1],output.mean(1)],dim=-1))\n\n    def forward(self,frames):\n        b,t,c,h,w=frames.shape\n        return self.classify_features(self.encode(frames.reshape(b*t,c,h,w)).reshape(b,t,512))\n\n\nclass TemporalFallClassifier:\n    def __init__(self,path:Path,device=\'cpu\'):\n        checkpoint=torch.load(path,map_location=\'cpu\',weights_only=True)\n        if checkpoint.get(\'architecture\')!=\'resnet18-gru-v1\':raise ValueError(\'Unsupported temporal fall model\')\n        self.model=TemporalFallNet();self.model.load_state_dict(checkpoint[\'state_dict\'])\n        self.model.to(device).eval();self.device=device\n        self.threshold=float(checkpoint[\'threshold\']);self.frames=deque(maxlen=16)\n        self.last_sample=None;self.window_s=2.0\n        self.mean=np.array([.485,.456,.406],np.float32)[:,None,None]\n        self.std=np.array([.229,.224,.225],np.float32)[:,None,None]\n\n    @torch.inference_mode()\n    def predict(self,bgr,now=None):\n        now=time.monotonic() if now is None else now\n        if self.last_sample is not None and (now<self.last_sample or now-self.last_sample>.5):\n            self.frames.clear();self.last_sample=None\n        if self.last_sample is not None and now-self.last_sample<.125:return None\n        rgb=Image.fromarray(np.ascontiguousarray(bgr[:,:,::-1])).resize((224,224),Image.Resampling.BILINEAR)\n        x=np.asarray(rgb,dtype=np.float32).transpose(2,0,1)/255\n        x=torch.from_numpy(np.ascontiguousarray((x-self.mean)/self.std))[None].to(self.device)\n        self.frames.append(self.model.encode(x)[0]);self.last_sample=now\n        if len(self.frames)<16:return None\n        logits=self.model.classify_features(torch.stack(list(self.frames))[None])\n        return float(logits.softmax(1)[0,1].cpu())\n',encoding='utf-8')

In [ ]:
manifests=list(Path('/kaggle/input').rglob('fall-manifest.jsonl'))
assert len(manifests)==1,'Attach LinQ Fall Expanded 2026 Dataset only.'
SOURCE=manifests[0].parent;DATA=Path('/tmp/linq-fall-data')
from concurrent.futures import ThreadPoolExecutor
files=[p for p in SOURCE.rglob('*') if p.is_file()]
def copy_one(source):
    target=DATA/source.relative_to(SOURCE);target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(source,target)
with ThreadPoolExecutor(max_workers=8) as pool:list(pool.map(copy_one,files))
for name in ['fall-summary.json','fall-manifest.jsonl','SOURCE-NOTICES.txt']:
    shutil.copyfile(DATA/name,OUTPUT/name)
print('FALL_DATA_READY',json.loads((DATA/'fall-summary.json').read_text())['splits'],flush=True)

In [ ]:
HANDLE='yathebest/linq-fall-checkpoints-20261024'
CPU_BUDGET=min(100,(datetime.fromisoformat(STOP_UTC).timestamp()-time.time())/60-10)
assert CPU_BUDGET>10,'Insufficient time remaining before the overnight deadline.'
backup=subprocess.Popen([sys.executable,str(SCRIPTS/'backup_training.py'),'--root',str(OUTPUT),'--handle',HANDLE])
code=124
try:
    training=subprocess.run([sys.executable,str(SCRIPTS/'train_fall_temporal.py'),'--fresh',
      '--data',str(DATA),'--output',str(OUTPUT),'--device','cpu','--batch','4','--cpu-threads','4',
      '--workers','2','--head-epochs','30','--finetune-epochs','4','--finetune-minutes','30',
      '--max-minutes',str(CPU_BUDGET)],timeout=(CPU_BUDGET+10)*60)
    code=training.returncode
except subprocess.TimeoutExpired:
    print('FALL_TRAINING_TIMEOUT; preserving available checkpoints.',flush=True)
finally:
    backup.terminate()
    try:backup.wait(timeout=10)
    except subprocess.TimeoutExpired:backup.kill()
print('FALL_TRAINING_EXIT',code,flush=True)

In [ ]:
try:
    subprocess.run([sys.executable,str(SCRIPTS/'backup_training.py'),'--root',str(OUTPUT),'--handle',HANDLE,'--once'],timeout=300)
except subprocess.TimeoutExpired:print('Final external backup timed out; local artifacts retained.',flush=True)
archive=WORK/'linq-fall-trained-models.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in OUTPUT.rglob('*'):
        if path.is_file():z.write(path,path.relative_to(OUTPUT))
print('DOWNLOAD',archive,flush=True)
assert code==0,'Training incomplete; inspect preserved checkpoints and logs.'